In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db3",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [29]:

instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'

importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])

In [30]:
assistant.chat()

Hallo! Ik kan je helpen met 'een reis plannen' of 'een afspraak maken voor een paspoort'. Je kan 'vorige' zeggen als je een stap terug wilt doen, 'volgende' als je naar de volgende stap wilt. Zeg 'herhaal' als je wilt dat ik een stap herhaal. Je kan altijd een vraag stellen. Zeg nu 'reis' of 'paspoort'.


You:  reis


MATCH ['Reis', 'Reis', 'Reis'] , DISTANCE -2.384185791015625e-07 , CAT nav , DO travel CONTEXT b INDEX 0
Agent: Ik ga je instrueren om een reis met het ov te plannen op 9292.nl. Stap 1: Waar begint je reis? Vul dat in bij 'Van'.


You:  volgende


MATCH ['Volgende', 'Volgende', 'Volgende'] , DISTANCE -3.5762786865234375e-07 , CAT nav , DO next step CONTEXT 1 INDEX 0
Agent: Waar wil je naartoe? Vul dat in bij 'Naar'.


...

MATCH ['Okee', 'Idee', 'Idee'] , DISTANCE -3.5762786865234375e-07 , CAT nav , DO Confirm CONTEXT 2 INDEX 1
Agent: Ik verstond 'okee', maar verwacht hier geen antwoord. Zeg 'volgende' als je naar de volgende instructie wil, of stel een vraag over de huidige instructie.


You:  klaar


MATCH ['Klaar', 'Klaar', 'Klaar'] , DISTANCE 2.384185791015625e-07 , CAT nav , DO Done CONTEXT 2 INDEX 1
Agent: Je zegt 'klaar'. Weet je zeker dat je wil stoppen met het gesprek?


You:  ja


MATCH ['Ja', 'Ja', 'Ja'] , DISTANCE -4.76837158203125e-07 , CAT nav , DO Confirm CONTEXT q INDEX 1
Agent: Tot ziens!
